# تست پکیج Persian OCR در Colab

پکیج همان مسیر Paddle → Hezar و اصلاح شرطی نسخهٔ ۱۳ را اجرا می‌کند. ZIP سورس `persian-ocr-0.1.1.zip` را در سلول بعد آپلود کنید. Python 3.12 جدا ساخته می‌شود و نسخهٔ kernel اصلی مهم نیست.

اگر تصویر خودِ کارت آماده و درست‌جهت‌شده است، `RAW_PHOTO=False`. اگر عکس صحنه است، `RAW_PHOTO=True`؛ در این حالت ZIP سورس CardID را هم باید نصب کنید و runtime Colab باید GPU داشته باشد. بخش OCR پیش‌فرض CPU است.

خروجی memory فقط دیکشنری نتیجه را به kernel برمی‌گرداند؛ json فایل داده می‌نویسد؛ report گزارش HTML و باکس‌ها را هم می‌سازد. دانلود کش مدل‌ها مستقل از خروجی نتیجه است.

In [ ]:
RAW_PHOTO = False  # @param {type:"boolean"}
OUTPUT_MODE = "report"  # @param ["memory", "json", "report"]
QUALITY_MODE = "auto"  # @param ["auto", "off", "force"]


In [ ]:
import json
import os
from pathlib import Path, PurePosixPath
import stat
import subprocess
import sys
import uuid
import zipfile
from google.colab import files
from IPython.display import HTML, JSON, display

ROOT = Path.cwd() / "persian-ocr-lab"
ROOT.mkdir(parents=True, exist_ok=True)
RUN = ROOT / uuid.uuid4().hex[:8]
RUN.mkdir()

def safe_extract(data, target):
    from io import BytesIO
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(BytesIO(data)) as archive:
        if sum(m.file_size for m in archive.infolist()) > 2 * 1024**3:
            raise ValueError("ZIP too large")
        for member in archive.infolist():
            rel = PurePosixPath(member.filename.replace("\\", "/"))
            if rel.is_absolute() or ".." in rel.parts or any(":" in p for p in rel.parts) or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError("Unsafe ZIP member")
            path = target.joinpath(*rel.parts).resolve()
            if not path.is_relative_to(target.resolve()):
                raise ValueError("Unsafe ZIP path")
            if member.is_dir():
                path.mkdir(parents=True, exist_ok=True)
            else:
                path.parent.mkdir(parents=True, exist_ok=True)
                path.write_bytes(archive.read(member))

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("فقط ZIP سورس Persian OCR را انتخاب کنید.")
safe_extract(next(iter(uploaded.values())), RUN / "package")
projects = [p.parent for p in (RUN / "package").rglob("pyproject.toml")
            if (p.parent / "src" / "persian_ocr").is_dir()]
if len(projects) != 1:
    raise ValueError("ZIP باید سورس همین پکیج را داشته باشد.")
PROJECT = projects[0]
print(PROJECT)


In [ ]:
worker_env = os.environ.copy()
worker_env.pop("PYTHONHOME", None)
worker_env.pop("PYTHONPATH", None)
worker_env.update(PYTHONUTF8="1", PYTHONNOUSERSITE="1", MPLBACKEND="Agg")
subprocess.run([sys.executable, str(PROJECT / "scripts" / "setup_runtime.py"),
                "--root", str(ROOT / "runtime"), "--torch-index", "cu126" if RAW_PHOTO else "cpu"],
               env=worker_env, check=True)
PYTHON = ROOT / "runtime" / "env" / "bin" / "python"
print("Package Python:", PYTHON)


## نصب اختیاری CardID

فقط در حالت عکس خام، ZIP سورس پکیج cardid-extractor را انتخاب کنید. نتیجهٔ not_found وارد OCR نمی‌شود؛ needs_review در صورت داشتن تصویر کارت، وارد OCR می‌شود و هشدار بررسی در گزارش حفظ می‌شود.

In [ ]:
if RAW_PHOTO:
    uploaded_cardid = files.upload()
    if len(uploaded_cardid) != 1:
        raise ValueError("فقط ZIP سورس CardID را انتخاب کنید.")
    safe_extract(next(iter(uploaded_cardid.values())), RUN / "cardid")
    projects = [p.parent for p in (RUN / "cardid").rglob("pyproject.toml")
                if (p.parent / "src" / "cardid").is_dir()]
    if len(projects) != 1:
        raise ValueError("CardID source project not found")
    subprocess.run([str(PYTHON), "-m", "pip", "install", str(projects[0])], env=worker_env, check=True)
    subprocess.run([str(PYTHON), "-m", "pip", "check"], env=worker_env, check=True)
else:
    print("Aligned card input: CardID is not needed.")


## عکس و اجرای پکیج

در این مثال یک عکس در هر اجرا می‌خوانیم. متغیر data پس از اجرا دیکشنری قابل استفاده در بقیهٔ کد است.

In [ ]:
uploaded_image = files.upload()
if len(uploaded_image) != 1:
    raise ValueError("یک عکس را انتخاب کنید.")
name, content = next(iter(uploaded_image.items()))
image_path = RUN / Path(name).name
image_path.write_bytes(content)
command = [str(PYTHON), "-m", "persian_ocr.cli", str(image_path),
           "--quality", QUALITY_MODE, "--detailed"]
if RAW_PHOTO:
    command += ["--raw-photo"]
if OUTPUT_MODE in ("json", "report"):
    command += ["--json", str(RUN / "data.json")]
if OUTPUT_MODE == "report":
    command += ["--report", str(RUN / "report")]
process = subprocess.run(command, env=worker_env, capture_output=True, text=True, encoding="utf-8")
(RUN / "runtime.log").write_text(process.stderr, encoding="utf-8")
if process.stderr:
    print(process.stderr[-6000:])
try:
    payload = json.loads(process.stdout)
except json.JSONDecodeError as exc:
    print(process.stdout[-6000:])
    raise RuntimeError("Worker output was not valid JSON; see runtime.log") from exc
if process.returncode:
    raise RuntimeError(payload)
data = payload["data"]
print("Status:", payload["status"])
display(JSON(data, expanded=True))
report_path = RUN / "report" / "report.html"
if report_path.exists():
    display(HTML(report_path.read_text(encoding="utf-8")))


In [ ]:
# استفاده در کدهای بعدی
print(data["first_name"])
print(data["national_id"])

# دانلود اختیاری خروجی‌های ذخیره‌شده
if OUTPUT_MODE == "report":
    bundle = RUN / "ocr-report.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in (RUN / "report").rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(RUN))
        archive.write(RUN / "runtime.log", "runtime.log")
    files.download(str(bundle))
elif OUTPUT_MODE == "json":
    files.download(str(RUN / "data.json"))
